# Test token-length displacement

Compare head-1 output geometry against token length, cumulative count, and line position.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = next(
    candidate for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]
    if (candidate / "results/line_breaks_results").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks/06_layer1_mechanisms"))
from common import *
EXAMPLE_SEED=6001
ctx=load_context(EXAMPLE_SEED)
model,vocab,batch,cfg,device=ctx.model,ctx.vocab,ctx.batch,ctx.cfg,ctx.device
tokens=batch.tokens[0].detach().cpu().numpy()
print("checkpoint",ctx.checkpoint["step"],"seed",EXAMPLE_SEED,"line width",int(batch.line_lengths[0]))
print("newline positions",newline_positions(tokens,vocab))

/Users/rohanbuluswar/Desktop/Line Break Project/.venv_new/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Building vocab...
('BOS', '_NEWLINE_')
checkpoint 10000 seed 6001 line width 89
newline positions [22, 44, 66, 87, 106, 129, 149]


In [2]:
name="blocks.1.attn.hook_z"
_,c=model.run_with_cache(batch.tokens,prepend_bos=False,names_filter=[name])
z=c[name][0,:,1].detach().cpu().numpy()
lengths=np.array([token_length(t,vocab) for t in tokens]); counts=cumulative_counts(tokens,vocab); pos=positions_since_newline(tokens,vocab)
for label,x in [("token length",lengths),("cumulative count",counts),("segment position",pos)]:
 v=np.isfinite(x); print(label,"r",float(np.corrcoef(x[v],z[v,0])[0,1]))
print(pd.DataFrame(z).assign(length=lengths).groupby("length").mean().round(3))
plot_scalar_by_position(z[:,0],lengths,"Layer-1 head 1 output coordinate 0","figures/layer1_head1_length_displacement.png")

token length r -0.033592049567268294
cumulative count r -0.6878501191978291
segment position r -0.7336467551072945
            0      1      2      3      4      5      6      7      8      9  \
length                                                                         
0.0     0.995 -1.012 -1.229  1.321 -0.109 -0.009 -0.914  0.350  2.319 -2.230   
1.0     0.636 -0.755 -0.247 -0.291 -0.236 -0.971  0.068 -0.014  0.865 -1.232   
2.0     1.026 -1.015 -0.771  0.497 -0.467 -0.467 -0.403  0.379  1.569 -1.802   
3.0     0.996 -0.936 -0.533  0.177 -0.464 -0.661 -0.193  0.320  1.287 -1.612   
4.0     0.869 -0.838 -0.385 -0.052 -0.390 -0.797 -0.049  0.207  1.082 -1.436   
5.0     0.803 -0.803 -0.308 -0.169 -0.355 -0.873  0.017  0.146  0.976 -1.349   
6.0     1.097 -0.855 -0.403  0.051 -0.550 -0.688 -0.071  0.430  1.168 -1.546   
7.0     0.582 -0.764 -0.283 -0.275 -0.203 -0.975  0.052 -0.062  0.889 -1.231   
8.0     0.785 -0.883 -0.481  0.029 -0.327 -0.780 -0.119  0.127  1.168 -1.477   

    